<a href="https://colab.research.google.com/github/Titir1/codes-for-time-series/blob/master/Pricing_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# Install

In [2]:
!pip install openai scikit-learn pandas numpy

In [ ]:
#Imports

In [23]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
import google.generativeai as genai
from google.colab import userdata

# Configure Gemini API
GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
genai.configure(api_key=GOOGLE_API_KEY)

gemini_model = genai.GenerativeModel('gemini-3-flash-preview') # Using 'gemini-3-flash-preview' as a general-purpose model

np.random.seed(42)

In [5]:
customers = []

for i in range(50):
    customers.append({
        "customer_id": f"C{i}",
        "customer_name": f"Customer_{i}",
        "segment": np.random.choice(["Premium","Standard","Competitor"]),
        "region": np.random.choice(["N_AMERICA","EMENA","APAC","CIS"]),
        "customer_value": np.random.uniform(10000,100000)
    })

customers_df = pd.DataFrame(customers)

In [ ]:
#Product + SKU

In [6]:
product_catalog = {
    "Smartphone": ["128GB","256GB","512GB"],
    "Laptop": ["i5","i7","i9"],
    "TV": ["32 inch","50 inch","65 inch"],
    "Air Conditioner": ["1 Ton","1.5 Ton","2 Ton"],
    "Accessory": ["Basic","Advanced"],
    "Networking Equipment": ["Small","Medium","Large"]
}

In [ ]:
#Cost

In [7]:
def get_cost(product, subgroup):

    base_prices = {
        "Smartphone": 300,
        "Laptop": 700,
        "TV": 400,
        "Air Conditioner": 500,
        "Accessory": 100,
        "Networking Equipment": 600
    }

    multiplier = {
        "128GB":1.0,"256GB":1.2,"512GB":1.5,
        "i5":1.0,"i7":1.3,"i9":1.6,
        "32 inch":1.0,"50 inch":1.3,"65 inch":1.6,
        "1 Ton":1.0,"1.5 Ton":1.3,"2 Ton":1.6,
        "Basic":1.0,"Advanced":1.5,
        "Small":1.0,"Medium":1.3,"Large":1.6
    }

    return base_prices[product] * multiplier[subgroup]

In [ ]:
#Elasticity

In [8]:
def subgroup_elasticity(product, subgroup):

    if subgroup in ["128GB","1 Ton","Basic","Small"]:
        return np.random.uniform(-2.5,-1.8)

    elif subgroup in ["512GB","2 Ton","Advanced","Large"]:
        return np.random.uniform(-1.0,-0.5)

    else:
        return np.random.uniform(-1.5,-1.0)

In [ ]:
#Season

In [9]:
def get_season(month):
    if month in [12,1,2]: return "Winter"
    elif month in [3,4,5]: return "Spring"
    elif month in [6,7,8]: return "Summer"
    else: return "Fall"

In [ ]:
# Region Tariff

In [10]:
region_tariff = {
    "N_AMERICA":0.25,
    "EMENA":0.18,
    "APAC":0.12,
    "CIS":0.20
}

In [ ]:
#Data Generation

In [11]:
data = []

for i in range(4000):

    cust = customers_df.sample(1).iloc[0]

    product = np.random.choice(list(product_catalog.keys()))
    subgroup = np.random.choice(product_catalog[product])

    elasticity = subgroup_elasticity(product, subgroup)

    month = np.random.randint(1,13)
    season = get_season(month)

    region = cust["region"]

    cost = get_cost(product, subgroup)
    tariff = region_tariff[region]

    inventory = np.random.randint(50,1000)
    quantity = np.random.randint(50,1000)

    margin = np.random.uniform(0.20,0.35)

    price = (cost*(1+tariff))/(1-margin)

    region_factor = {"N_AMERICA":1.2,"EMENA":1.0,"APAC":1.3,"CIS":0.9}[region]

    demand = 500 * region_factor * (price**elasticity)

    data.append([
        cust["customer_name"],cust["segment"],cust["customer_value"],region,
        product,subgroup,month,season,quantity,
        cost,tariff,elasticity,inventory,
        price,demand,margin
    ])

df = pd.DataFrame(data,columns=[
    "customer","segment","customer_value","region",
    "product","subgroup","month","season","quantity",
    "cost","tariff","elasticity","inventory",
    "price","demand","margin"
])

In [ ]:
#Seasonality

In [12]:
avg = df.groupby(["region","product","subgroup"])["demand"].mean().reset_index()
season_avg = df.groupby(["region","product","subgroup","season"])["demand"].mean().reset_index()

seasonality = season_avg.merge(avg,on=["region","product","subgroup"])
seasonality["mult"] = seasonality["demand_x"]/seasonality["demand_y"]

season_dict = {(r["region"],r["product"],r["subgroup"],r["season"]):r["mult"] for _,r in seasonality.iterrows()}

def get_season_multiplier(region,product,subgroup,season):
    return season_dict.get((region,product,subgroup,season),1.0)

In [ ]:
#Model

In [13]:
df["season_factor"] = df.apply(
    lambda x:get_season_multiplier(x["region"],x["product"],x["subgroup"],x["season"]),axis=1
)

X = df[[
    "cost","tariff","elasticity","inventory","demand",
    "season_factor","quantity","customer_value"
]]

y = df["margin"]

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2)

model = RandomForestRegressor(n_estimators=120)
model.fit(X_train,y_train)

RandomForestRegressor(n_estimators=120)

In [14]:
def segment_rules(segment):

    if segment == "Premium":
        return {
            "margin_min": 0.20,
            "margin_target": 0.28,
            "max_discount": 0.25,
            "strategy": "stable"
        }

    elif segment == "Standard":
        return {
            "margin_min": 0.22,
            "margin_target": 0.32,
            "max_discount": 0.12,
            "strategy": "competitive"
        }

    else:  # Competitor
        return {
            "margin_min": 0.30,
            "margin_target": 0.40,
            "max_discount": 0.03,
            "strategy": "high_price"
        }

In [ ]:
#Pricing Engine

In [15]:
def dynamic_price(row, pred_margin):

    rules = segment_rules(row["segment"])

    # Apply segment margin control
    margin = max(rules["margin_min"],
                 min(pred_margin, rules["margin_target"]))

    cost_price = (row["cost"]*(1+row["tariff"]))/(1-margin)

    sf = get_season_multiplier(
        row["region"], row["product"], row["subgroup"], row["season"]
    )

    comp = row["cost"] * np.random.uniform(1.1,1.4)
    cap = np.random.uniform(0.9,1.1)

    base = cost_price * sf * cap

    # 🔥 Segment-based pricing strategy
    if rules["strategy"] == "stable":
        price = 0.7*(row["cost"]*1.3) + 0.3*base

    elif rules["strategy"] == "competitive":
        price = (base + comp) / 2

    else:  # high_price
        price = base * 1.15

    trace = {
        "margin_used": margin,
        "season_factor": sf,
        "elasticity": row["elasticity"],
        "capacity_factor": cap,
        "competitor_price": comp,
        "segment": row["segment"]
    }

    return price, trace

In [ ]:
#Discount

In [16]:
def optimize_discount(row, pred_margin):

    rules = segment_rules(row["segment"])

    base_price, trace = dynamic_price(row, pred_margin)

    # 🔥 Segment-based discount logic
    if row["quantity"] > 500:
        discount = rules["max_discount"]
    else:
        discount = rules["max_discount"] * 0.5

    final_price = base_price * (1 - discount)

    trace["discount"] = discount
    trace["segment"] = row["segment"]

    return discount, final_price, trace

In [ ]:
#Negotiation

In [17]:
def get_negotiation_range(price, segment):

    if segment == "Premium":
        floor = price * 0.90
        ceiling = price * 1.05

    elif segment == "Standard":
        floor = price * 0.92
        ceiling = price * 1.08

    else:  # Competitor
        floor = price * 0.95
        ceiling = price * 1.12

    return {
        "floor_price": floor,
        "target_price": price,
        "ceiling_price": ceiling
    }

In [ ]:
# AI Explanation

In [25]:
def explain(row,price,discount):

    prompt = f"""
    Explain pricing for Sunshine.

    Region: {row['region']}
    Product: {row['product']}
    Variant: {row['subgroup']}
    Segment: {row['segment']}

    Price: {price}
    Discount: {discount}

    Explain reasoning clearly.
    """

    res = gemini_model.generate_content(prompt)

    return res.text

In [ ]:
#RFQ Line with scenarion

In [28]:
def rfq_line(customer,product,subgroup,month,inventory,quantity,tariff_override=None):

    region = customer["region"]
    tariff = tariff_override if tariff_override is not None else region_tariff[region]

    cost = get_cost(product, subgroup)
    elasticity = subgroup_elasticity(product, subgroup)
    season = get_season(month)

    row = {
        "customer":customer["customer_name"],
        "segment":customer["segment"],
        "customer_value":customer["customer_value"],
        "region":region,
        "product":product,
        "subgroup":subgroup,
        "season":season,
        "cost":cost,
        "tariff":tariff,
        "elasticity":elasticity,
        "inventory":inventory,
        "quantity":quantity,
        "demand":500*(cost**elasticity)
    }

    # Prepare input for prediction as a DataFrame with correct column names
    predict_input = pd.DataFrame([[cost,tariff,elasticity,inventory,row["demand"],
                                   get_season_multiplier(region,product,subgroup,season),
                                   quantity,customer["customer_value"]]],
                                 columns=X.columns) # Use X.columns for feature names

    pred_margin = model.predict(predict_input)[0]

    discount,price,trace = optimize_discount(row,pred_margin)
    negotiation = get_negotiation_range(price, row["segment"])

    explanation = explain(row, price, discount) # Re-enable AI explanation

    return {
        "product":product,
        "subgroup":subgroup,
        "price":price,
        "discount":discount,
        "trace":trace,
        "explanation":explanation
    }

In [ ]:
#Scenario Engine

In [20]:
def run_scenarios(customer, base_item, scenarios):

    results = []

    for sc in scenarios:

        tariff = sc.get("tariff", None)
        quantity = sc.get("quantity", base_item["quantity"])

        res = rfq_line(
            customer,
            product=base_item["product"],
            subgroup=base_item["subgroup"],
            month=base_item["month"],
            inventory=base_item["inventory"],
            quantity=quantity,
            tariff_override=tariff
        )

        res["scenario"] = sc
        results.append(res)

    return pd.DataFrame(results)

In [ ]:
#Multi-LineRFQ

In [21]:
def multi_line_rfq(customer,rfq_items):

    results = []
    total_value = 0

    for item in rfq_items:

        res = rfq_line(
            customer,
            item["product"],
            item["subgroup"],
            item["month"],
            item["inventory"],
            item["quantity"]
        )

        value = res["price"] * item["quantity"]
        total_value += value

        res["quantity"] = item["quantity"]
        res["line_value"] = value

        results.append(res)

    return {"line_items":results,"total_quote_value":total_value}

In [ ]:
#Run Scenario

In [22]:
cust = customers_df.sample(1).iloc[0]

base_item = {
    "product":"Air Conditioner",
    "subgroup":"2 Ton",
    "month":7,
    "inventory":800,
    "quantity":500
}

scenarios = [
    {"tariff":0.10},
    {"tariff":0.20},
    {"tariff":0.30}
]

run_scenarios(cust, base_item, scenarios)

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
ERROR:tornado.access:503 POST /v1beta/models/gemini-3-flash-preview:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 26687.83ms
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(


,product,subgroup,price,discount,trace,explanation,scenario
0,Air Conditioner,2 Ton,1952.336335,0.015,"{'margin_used': 0.3, 'season_factor': 1.379277...",Based on the data provided for the **Sunshine ...,{'tariff': 0.1}
1,Air Conditioner,2 Ton,2268.899579,0.015,"{'margin_used': 0.3, 'season_factor': 1.379277...","Based on the data provided, here is the pricin...",{'tariff': 0.2}
2,Air Conditioner,2 Ton,2134.736038,0.015,"{'margin_used': 0.3, 'season_factor': 1.379277...",Based on the data provided for the **Sunshine ...,{'tariff': 0.3}


In [27]:
import ipywidgets as widgets
from IPython.display import display, HTML

# Dropdown for Customer Selection
customer_names = customers_df['customer_name'].tolist()
customer_selector = widgets.Dropdown(
    options=customer_names,
    value=customer_names[0],
    description='Customer:',
    disabled=False,
)

# Dropdown for Product Selection
product_names = list(product_catalog.keys())
product_selector = widgets.Dropdown(
    options=product_names,
    value=product_names[0],
    description='Product:',
    disabled=False,
)

# Dropdown for Subgroup Selection (will be updated dynamically)
subgroup_selector = widgets.Dropdown(
    options=product_catalog[product_selector.value],
    value=product_catalog[product_selector.value][0],
    description='Subgroup:',
    disabled=False,
)

# Text input for Month (assuming integer 1-12)
month_input = widgets.IntSlider(
    value=7,
    min=1,
    max=12,
    step=1,
    description='Month:',
    disabled=False,
    continuous_update=False,
    orientation='horizontal',
    readout=True,
    readout_format='d'
)

# Text input for Inventory
inventory_input = widgets.IntSlider(
    value=500,
    min=0,
    max=2000,
    step=10,
    description='Inventory:',
    disabled=False,
    continuous_update=False,
    orientation='horizontal',
    readout=True,
    readout_format='d'
)

# Text input for Quantity
quantity_input = widgets.IntSlider(
    value=100,
    min=1,
    max=1000,
    step=5,
    description='Quantity:',
    disabled=False,
    continuous_update=False,
    orientation='horizontal',
    readout=True,
    readout_format='d'
)

# Output widget to display results
output_display = widgets.Output()

# Function to update subgroup options based on selected product
def update_subgroup_options(*args):
    subgroup_selector.options = product_catalog[product_selector.value]
    subgroup_selector.value = product_catalog[product_selector.value][0]

product_selector.observe(update_subgroup_options, 'value')

def generate_pricing(customer_name, product, subgroup, month, inventory, quantity):
    with output_display:
        output_display.clear_output()
        selected_customer = customers_df[customers_df['customer_name'] == customer_name].iloc[0]

        res = rfq_line(
            selected_customer,
            product=product,
            subgroup=subgroup,
            month=month,
            inventory=inventory,
            quantity=quantity
        )

        print(f"Product: {res['product']} - {res['subgroup']}")
        print(f"Calculated Price: {res['price']:.2f}")
        print(f"Discount Applied: {res['discount']*100:.2f}%")
        print("\n--- Pricing Explanation ---")
        print(res['explanation'])

# Create interactive dashboard
interactive_dashboard = widgets.interactive(
    generate_pricing,
    customer_name=customer_selector,
    product=product_selector,
    subgroup=subgroup_selector,
    month=month_input,
    inventory=inventory_input,
    quantity=quantity_input
)

display(interactive_dashboard, output_display)

interactive(children=(Dropdown(description='Customer:', options=('Customer_0', 'Customer_1', 'Customer_2', 'Cu…

Output()